In [6]:
import asyncio # Imports the library for running asynchronous code.
from agents import Agent, Runner, set_tracing_disabled # Imports the core components from the OpenAI Agents SDK.
import os 
# Disable the SDK's tracing feature to keep the output clean for this tutorial.
set_tracing_disabled(True)

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
# Define a simple agent for a quick test.
agent = Agent(
    name="Assistant", # Assign a name to the agent.
    instructions="Reply very concisely.", # Provide a simple instruction to guide its behavior.
    # Specify the model using the LiteLLM provider format for Nebius.
    model="litellm/nebius/moonshotai/Kimi-K2-Instruct",
)

# Execute the agent with a test prompt. 'await' is used because the run is an asynchronous operation.
result = await Runner.run(
    agent, # The agent to run.
    "Tell me why it is important to evaluate AI agents." # The user's input prompt.
)

# Print the final text output from the agent's response.
print(result.final_output)

Evaluation ensures AI agents are safe, reliable, and aligned with human goals by identifying failures, biases, and misuse risks before deployment.


In [27]:
from openai import OpenAI 

client = OpenAI(
    # Sets the base URL for the API endpoint to the Nebius service.
    base_url=os.environ.get("LITELLM_API_BASE_URL"),
    # Sets the API key for authentication. Replace with your actual key, preferably loaded from a secure source.
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [7]:
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class MemoryNote:
    text: str
    last_update_date: str
    keywords: List[str]


In [10]:
@dataclass
class TravelState:
    profile: Dict[str, Any] = field(default_factory=dict)

    global_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    session_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    trip_history: Dict[str, Any] = field(default_factory=lambda: {"trips":[]})

    system_frontmatter: str = ""
    global_memories_md: str = ""
    session_memories_md:str = ""

    inject_session_memories_next_turn: bool = False

In [13]:
user_state = TravelState(
    profile={
        "global_customer_id": "crm_12345",
        "name": "John Doe",
        "age": "31",
        "home_city": "San Francisco",
        "currency" : "USD",
        "passport_expiry_date": "2029-06-12",
        "loyalty_status": {"airline": "United Gold", "hotel": "Marriott Titanium"},
        "loyalty_ids": {"marriott": "MR998877", "hilton": "HH445566", "hyatt": "HY112233"},
        "seat_preference": "aisle",
        "tone": "concise and friendly",
        "active_visas": ["Schengen", "US"],
        "insurance_coverage_profile": {
            "car_rental": "primary_cdw_included",
            "travel_medical": "covered",
        },
    },
    global_memory = {
        "notes": [
            # Each note is an instance of MemoryNote, converted to a dictionary for JSON compatibility.
            MemoryNote(
                text="For trips shorter than a week, user generally prefers not to check bags.",
                last_update_date="2025-04-05",
                keywords=["baggage", "short_trip"],
            ).__dict__,
            MemoryNote(
                text="User usually prefers aisle seats.",
                last_update_date="2024-06-25",
                keywords=["seat_preference"],
            ).__dict__,
            MemoryNote(
                text="User generally likes central, walkable city-center neighborhoods.",
                last_update_date="2024-02-11",
                keywords=["neighborhood"],
            ).__dict__,
            MemoryNote(
                text="User generally likes to compare options side-by-side",
                last_update_date="2023-02-17",
                keywords=["pricing"],
            ).__dict__,
            MemoryNote(
                text="User prefers high floors",
                last_update_date="2023-02-11",
                keywords=["room"],
            ).__dict__,
        ]
    },
    trip_history = {
        "trips": [
            {
                # Core trip details
                "from_city": "Istanbul",
                "from_country": "Turkey",
                "to_city": "Paris",
                "to_country": "France",
                "check_in_date": "2025-05-01",
                "check_out_date": "2025-05-03",
                "trip_purpose": "leisure",  # leisure | business | family | etc.
                "party_size": 1,

                # Flight details
                "flight": {
                    "airline": "United",
                    "airline_status_at_booking": "United Gold",
                    "cabin_class": "economy_plus",
                    "seat_selected": "aisle",
                    "seat_location": "front",          # front | middle | back
                    "layovers": 1,
                    "baggage": {"checked_bags": 0, "carry_ons": 1},
                    "special_requests": ["vegetarian_meal"],  # optional
                },

                # Hotel details
                "hotel": {
                    "brand": "Hilton",
                    "property_name": "Hilton Paris Opera",
                    "neighborhood": "city_center",
                    "bed_type": "king",
                    "smoking": "non_smoking",
                    "high_floor": True,
                    "early_check_in": False,
                    "late_check_out": True,
                },
            }
        ]
    },
)

In [15]:
from datetime import datetime,timezone
from typing import List
from agents import function_tool,RunContextWrapper


def _today_iso_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT")


@function_tool
def save_memory_note(ctx: RunContextWrapper[TravelState],text:str, keywords = List[str]) -> dict:
    if "notes" not in ctx.context.session_memory or ctx.context.session_memory["notes"] is None:
        ctx.context.session_memory["notes"] = []

    clean_keywords = [
        k.strip().lower()
        for k in keywords if isinstance(k,str) and k.strip()
        ][:3]

    ctx.context.session_memory["notes"].append({
        "text": text.strip(),
        "last_update_date": _today_iso_utc(),
        "keywords": clean_keywords
    })
    return {"ok":True}


/Users/hamza/Desktop/PROJECTS/contextual-engineering-ai-agents/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value typing.List[str] is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [16]:
from __future__ import annotations
import asyncio
from collections import deque
from typing import Any, Deque, List, cast
from agents.memory.session import SessionABC
from agents.items import TResponse, TResponseInputItem

ROLE_USER = "user"

def _is_user_msg(item: TResponseInputItem) -> bool:
    if isinstance(item,dict):
        role = item.get('role')
        if role is not None:
            return role == ROLE_USER

        if item.get("type") == "message":
            return item.get("role") == ROLE_USER

    return getattr(item, "role", None) == ROLE_USER

class TrimmingSession(SessionABC):

    def __init__(self, session_id: str, state: TravelState, max_turns: int = 8):
        self.session_id = session_id
        self.state = state
        self.max_turns = max(1, int(max_turns))
        self._items : Deque[TResponseInputItem] = deque()
        self._lock = asyncio.Lock()

    async def get_items(self, limit:int | None = None) -> List[TResponseInputItem]:
        async with self._lock:
            trimmed = self._trim_to_last_turns(list(self._items))
            return trimmed[-limit:] if (limit is not None and limit >=0) else trimmed

    async def add_items(self, items: List[TResponseInputItem]) -> None:
        """"""
        if not items:
            return

        async with self._lock:
            self._items.extend(items)
            original_len = len(self._items)
            trimmed = self._trim_to_last_turns(list(self._items))

            if len(trimmed) < original_len:
                self.state.inject_session_memories_next_turn = True

            self._items.clear()
            self._items.extend(trimmed)

    async def pop_item(self):
        async with self._lock:
            return self._items.pop()

    async def clear_session(self) -> None:
        async with self._lock:
            self._items.clear()

    async def _trim_to_last_turns(self,items: List[TResponseInputItem]) -> List[TResponseInputItem]:
        if not items:
            return items

        count = 0
        start_idx = 0

        for i in range(len(items) - 1, -1, -1):
            if _is_user_msg(items[i]):
                count +=1
                if count == self.max_turns:
                    start_idx = i
                    break 
        return items[start_idx:]
        





In [17]:
session = TrimmingSession("my_session", user_state, max_turns=20)

In [18]:
# Define a multi-line string containing the rules and guidelines for how the agent should use memory.
MEMORY_INSTRUCTIONS = """
<memory_policy>
You may receive two memory lists:
- GLOBAL memory = long-term defaults (“usually / in general”).
- SESSION memory = trip-specific overrides (“this trip / this time”).

How to use memory:
- Use memory only when it is relevant to the user’s current decision (flight/hotel/insurance choices).
- Apply relevant memory automatically when setting tone, proposing options and making recommendations.
- Do not repeat memory verbatim to the user unless it’s necessary to confirm a critical constraint.

Precedence and conflicts:
1) The user’s latest message in this conversation overrides everything.
2) SESSION memory overrides GLOBAL memory for this trip when they conflict.
   - Example: GLOBAL “usually aisle” + SESSION “this time window to sleep” ⇒ choose window for this trip.
3) Within the same memory list, if two items conflict, prefer the most recent by date.
4) Treat GLOBAL memory as a default, not a hard constraint, unless the user explicitly states it as non-negotiable.

When to ask a clarifying question:
- Ask exactly one focused question only if a memory materially affects booking and the user’s intent is ambiguous.
  (e.g., “Do you want to keep the window seat preference for all legs or just the overnight flight?”)

Where memory should influence decisions (check these before suggesting options):
- Flights: seat preference, baggage habits (carry-on vs checked), airline loyalty/status, layover tolerance if mentioned.
- Hotels: neighborhood/location style (central/walkable), room preferences (high floor), brand loyalty IDs/status.
- Insurance: known coverage profile (e.g., CDW included) and whether the user wants add-ons this trip.

Memory updates:
- Do NOT treat “this time” requests as changes to GLOBAL defaults.
- Only promote a preference into GLOBAL memory if the user indicates it’s a lasting rule
  (e.g., “from now on”, “generally”, “I usually prefer X now”).
- If a new durable preference/constraint appears, store it via the memory tool (short, general, non-PII).

Safety:
- Never store or echo sensitive PII (passport numbers, payment details, full DOB).
- If a memory seems stale or conflicts with user intent, defer to the user and proceed accordingly.
</memory_policy>
"""

In [19]:
# Define a multi-line string containing the rules and guidelines for how the agent should use memory.
MEMORY_INSTRUCTIONS = """
<memory_policy>
You may receive two memory lists:
- GLOBAL memory = long-term defaults (“usually / in general”).
- SESSION memory = trip-specific overrides (“this trip / this time”).

How to use memory:
- Use memory only when it is relevant to the user’s current decision (flight/hotel/insurance choices).
- Apply relevant memory automatically when setting tone, proposing options and making recommendations.
- Do not repeat memory verbatim to the user unless it’s necessary to confirm a critical constraint.

Precedence and conflicts:
1) The user’s latest message in this conversation overrides everything.
2) SESSION memory overrides GLOBAL memory for this trip when they conflict.
   - Example: GLOBAL “usually aisle” + SESSION “this time window to sleep” ⇒ choose window for this trip.
3) Within the same memory list, if two items conflict, prefer the most recent by date.
4) Treat GLOBAL memory as a default, not a hard constraint, unless the user explicitly states it as non-negotiable.

When to ask a clarifying question:
- Ask exactly one focused question only if a memory materially affects booking and the user’s intent is ambiguous.
  (e.g., “Do you want to keep the window seat preference for all legs or just the overnight flight?”)

Where memory should influence decisions (check these before suggesting options):
- Flights: seat preference, baggage habits (carry-on vs checked), airline loyalty/status, layover tolerance if mentioned.
- Hotels: neighborhood/location style (central/walkable), room preferences (high floor), brand loyalty IDs/status.
- Insurance: known coverage profile (e.g., CDW included) and whether the user wants add-ons this trip.

Memory updates:
- Do NOT treat “this time” requests as changes to GLOBAL defaults.
- Only promote a preference into GLOBAL memory if the user indicates it’s a lasting rule
  (e.g., “from now on”, “generally”, “I usually prefer X now”).
- If a new durable preference/constraint appears, store it via the memory tool (short, general, non-PII).

Safety:
- Never store or echo sensitive PII (passport numbers, payment details, full DOB).
- If a memory seems stale or conflicts with user intent, defer to the user and proceed accordingly.
</memory_policy>
"""

In [20]:
from pickle import TRUE
import yaml


def render_frontmatter(profile: dict) -> str:
    payload = {"profile":profile}
    y = yaml.safe_dump(payload, sort_keys = False).strip()
    return f"---\n{y}\n---"


def render_global_memories_md(global_notes: list[dict], k:int = 6) -> str:
    if not global_notes:
        return "- (none)"

    notes_sorted = sorted(global_notes, key=lambda n:n.get("last_update_date",""), reverse=True)
    top = notes_sorted[:k]
    return "\n".join([f"- {n["text"]}" for n in top])


def render_session_memories_md(session_notes: list[dict], k:int = 8) -> str:
    if not session_notes:
        return "- (none)"

    top = session_notes[-k:]
    return "\n".join([f"- {n['text']}" for n in top])

In [21]:
from agents import AgentHooks, Agent


class MemoryHooks(AgentHooks[TravelState]):

    def __init__(self, client: client):
        self.client = client

    async def on_start(self, ctx: RunContextWrapper[TravelState], agent:Agent) -> None:
        ctx.context.system_frontmatter = render_frontmatter(ctx.context.profile)
        ctx.context.global_memories_md = render_global_memories_md((ctx.context.global_memory or {}).get("notes", []))


        if ctx.context.inject_session_memories_next_turn:
            ctx.context.session_memories_md = render_session_memories_md(
                ctx.context.session_memory or {}
            ).get("notes",[])
        else:
            ctx.context.session_memories_md = ""


In [22]:
# Define the base system prompt for the travel concierge agent.
BASE_INSTRUCTIONS = f"""
You are a concise, reliable travel concierge. 
Help users plan and book flights, hotels, and car/travel insurance.\n\n
Guidelines:\n
- Collect key trip details and confirm understanding.\n
- Ask only one focused clarifying question at a time.\n
- Provide a few strong options with brief tradeoffs, then recommend one.\n
- Respect stable user preferences and constraints; avoid assumptions.\n
- Before booking, restate all details and get explicit approval.\n
- Never invent prices, availability, or policies—use tools or state uncertainty.\n
- Do not repeat sensitive PII; only request what is required.\n
- Track multi-step itineraries and unresolved decisions.\n\n
"""

In [23]:
async def instructions(ctx: RunContextWrapper[TravelState], agent:Agent) -> str:
    s = ctx.context

    if s.inject_session_memories_next_turn and not s.session_memories_md:
        s.session_memories_md = render_session_memories_md(
            (s.session_memory or {}).get("notes",[])
        )

    session_block = ""

    if s.inject_session_memories_next_turn and s.session_memories_md:
        session_block = (
            "\n\nSESSION memory (temporary: overrides GLOBAL when conflicting):\n"
            + s.session_memories_md
        )

        s.inject_session_memories_next_turn = False
        s.session_memories_md = ""

    return (
        BASE_INSTRUCTIONS
        +"\n\n<user_profile\n"+ (s.system_frontmatter or "") + "\n</user_profile>"
        +"\n\n<memories>\n"
        +"GLOBAL memory: \n" + (s.global_memories_md or "- (none)")
        +session_block
        +"\n</memories>"
        +"\n\n" + MEMORY_INSTRUCTIONS
    )

In [28]:
travel_concierage_agent = Agent(
    name= "Travel Concierage",
    model = "littlellm/nebius/moonshotai/Kimi-K2-Instruct",
    instructions = instructions,
    hooks = MemoryHooks(client),
    tools = [save_memory_note]
)